# Dataset Reconnaissance
Inspect the structure of each mounted dataset **without extracting**.
Run this once, push results to HuggingFace, then never re-run.

In [ ]:
# Cell 1: Setup
import os, json, zipfile, rarfile
from pathlib import Path
from collections import Counter, defaultdict

REPO_ID = "david-net-av/backup"  # HF repo for recon reports
KAGGLE_INPUT = Path("/kaggle/input")
OUT_DIR = Path("/kaggle/working/dataset_recon")
OUT_DIR.mkdir(exist_ok=True)

# Install rarfile if needed
!pip install -q rarfile 2>/dev/null
print("Setup done.")

In [ ]:
# Cell 2: Discover mounted datasets
mounted = []
for d in sorted(KAGGLE_INPUT.iterdir()):
    if d.is_dir():
        mounted.append(d)
        print(f"  {d.name}/")
print(f"\nFound {len(mounted)} mounted datasets.")

In [ ]:
# Cell 3: Recon functions
def recon_zip(path: Path) -> dict:
    """List contents of a zip without extracting."""
    try:
        with zipfile.ZipFile(path) as zf:
            entries = []
            for info in zf.infolist():
                entries.append({
                    "name": info.filename,
                    "size": info.file_size,
                    "compressed": info.compress_size,
                })
            return {"type": "zip", "entries": entries, "count": len(entries)}
    except Exception as e:
        return {"type": "zip", "error": str(e)}


def recon_rar(path: Path) -> dict:
    """List contents of a rar without extracting."""
    try:
        rf = rarfile.RarFile(path)
        entries = []
        for info in rf.infolist():
            entries.append({
                "name": info.filename,
                "size": info.file_size,
                "compressed": info.compress_size if hasattr(info, 'compress_size') else 0,
            })
        return {"type": "rar", "entries": entries, "count": len(entries)}
    except Exception as e:
        return {"type": "rar", "error": str(e)}


def build_tree(entries: list[dict]) -> dict:
    """Build directory tree from flat file list."""
    tree = {}
    exts = Counter()
    total_size = 0
    for e in entries:
        name = e["name"]
        size = e.get("size", 0)
        total_size += size
        ext = Path(name).suffix.lower()
        exts[ext] += 1
        parts = name.split("/")
        node = tree
        for p in parts[:-1]:
            if p not in node:
                node[p] = {}
            node = node[p]
    return {"tree": tree, "extensions": dict(exts), "total_size": total_size, "file_count": len(entries)}


def recon_archive(path: Path) -> dict:
    """Dispatch to zip or rar recon."""
    ext = path.suffix.lower()
    if ext == ".zip":
        return recon_zip(path)
    elif ext in (".rar", ".001"):
        return recon_rar(path)
    else:
        return {"type": "unknown", "error": f"unsupported extension: {ext}"}


print("Recon functions ready.")

In [ ]:
# Cell 4: Run recon on all mounted datasets
results = {}

for ds_dir in mounted:
    ds_name = ds_dir.name
    print(f"\n=== {ds_name} ===")
    
    # Scan for archives
    archives = []
    loose_files = []
    for f in ds_dir.rglob("*"):
        if f.is_file():
            if f.suffix.lower() in (".zip", ".rar") or (f.suffix.isdigit() and f.stem.split(".")[-1] == "001"):
                archives.append(f)
            else:
                loose_files.append({"name": str(f.relative_to(ds_dir)), "size": f.stat().st_size})
    
    report = {
        "name": ds_name,
        "path": str(ds_dir),
        "archives": [],
        "loose_files_count": len(loose_files),
        "loose_files_sample": loose_files[:20],
    }
    
    for arch in sorted(archives, key=lambda p: p.name)[:5]:  # limit to 5 archives
        print(f"  Scanning: {arch.name} ({arch.stat().st_size / 1e6:.1f} MB)")
        recon = recon_archive(arch)
        if "entries" in recon:
            tree_info = build_tree(recon["entries"])
            recon["tree"] = tree_info["tree"]
            recon["extensions"] = tree_info["extensions"]
            recon["total_size"] = tree_info["total_size"]
            recon["file_count"] = tree_info["file_count"]
            print(f"    -> {tree_info['file_count']} files, {tree_info['total_size'] / 1e9:.2f} GB")
            print(f"    Extensions: {tree_info['extensions']}")
        else:
            print(f"    -> ERROR: {recon.get('error', 'unknown')}")
        report["archives"].append({"name": arch.name, "size": arch.stat().st_size, "recon": recon})
    
    results[ds_name] = report
    print(f"  Loose files: {len(loose_files)}")

print(f"\nRecon complete for {len(results)} datasets.")

In [ ]:
# Cell 5: Save reports
for name, report in results.items():
    out_path = OUT_DIR / f"{name}.json"
    with open(out_path, "w") as f:
        json.dump(report, f, indent=2, default=str)
    print(f"Saved: {out_path}")

# Summary
print("\n=== SUMMARY ===")
for name, report in results.items():
    total_archives = sum(a["recon"].get("total_size", 0) for a in report["archives"] if "recon" in a)
    print(f"{name}: {len(report['archives'])} archives, {total_archives/1e9:.2f} GB compressed, {report['loose_files_count']} loose files")

In [ ]:
# Cell 6: Push to HuggingFace
from huggingface_hub import HfApi

hf_token = os.environ.get("HF_TOKEN") or os.environ.get("hf")
if not hf_token:
    print("ERROR: HF_TOKEN not found in Kaggle Secrets or env")
else:
    api = HfApi()
    # Ensure repo exists
    try:
        api.create_repo(REPO_ID, repo_type="model", token=hf_token, exist_ok=True)
    except Exception as e:
        print(f"Repo creation note: {e}")
    
    for f in OUT_DIR.glob("*.json"):
        api.upload_file(
            path_or_fileobj=str(f),
            path_in_repo=f"dataset_recon/{f.name}",
            repo_id=REPO_ID,
            repo_type="model",
            token=hf_token,
        )
        print(f"Pushed: {f.name}")
    
    print(f"\nAll reports pushed to {REPO_ID}/dataset_recon/")